# 3D Extension of the Hierarchical Stackelberg Security Problem

## Overall objective

Extend the validated 2D `(z, h)` Stackelberg security game in `p1b_4D` to a full 3D `(x, y, h)` setting: the Attacker's horizontal motion gains a second free dimension `y`, and a `heading` action controls direction within that horizontal plane. The Defender's sensor placement search becomes a 2D `(x_sensor, y_sensor)` search instead of 1D.

This is a **new, independent module set** (not a literal import of `p1b_4D`), because the state/action dimensionality change is pervasive (terrain, LOS geometry, the Bellman sweep direction, the action space, the Defender search space, and visualization all change shape). The goal is to replicate `p1b_4D`'s *architecture and validation philosophy* faithfully, not its code.

## Core structural principle (the key insight from this session's design discussion)

In 2D, `z` was the Bellman DP's monotonic sweep axis (it strictly increases every step, since `gamma` is always negative -- an unpowered glider cannot climb), and `h` was the one *free* dimension nested inside each `z`-slice.

In 3D, altitude `h` **still** strictly decreases every step -- that constraint comes from `gamma` alone and has nothing to do with horizontal heading. So **`h` keeps exactly the same role `z` used to have**: it is still the DP's monotonic sweep axis, and the backward-induction Bellman recursion is still solved in a single exact pass (no value iteration). What changes is that the *free* dimension nested inside each sweep-slice grows from one (`h`, inside each `z`-slice) to two (`x, y`, inside each `h`-slice).

Consequently `heading` can be a fully **unconstrained** third action dimension (alongside `v`, `gamma`) -- helical/spiral paths (e.g. circling inside a hill's radar shadow while shedding altitude) are legal and, in principle, can be optimal. No turn-rate limit is required to keep the DP exact; a turn-rate limit would instead force `heading` to become a *state* variable (since the feasible action set would depend on the previous heading), which is a separate, heavier design choice this v1 toy explicitly avoids.

## Toy scenario (v1)

- Launch: `(x, y, h) = (0, 0, 0)`
- Goal: `(x, y) = (2500, 0)`, `h_goal` terrain-following (same principle as `p1b_4D`'s `h_goal = terrain_height(z_goal)` fix)
- Terrain: **one** radially-symmetric Gaussian hill centered at `(x, y) = (1500, 0)` on an otherwise flat plane -- the direct 3D analog of `p1b_4D`'s original single-hill baseline (single obstacle first; multi-hill generalization comes later, once this is validated, mirroring how the 2D work proceeded)
- Sensor: continuous 2D search over `(x_sensor, y_sensor)`, analogous to 2D's 1D continuous `z_sensor` DIRECT search

## Pipeline scheme (mirrors `p1b_4D`'s 4D J-map -> 2D projection -> 2D cost-to-go -> Bellman, becoming 6D -> 3D -> 3D -> Bellman)

| Phase | 2D (`p1b_4D`) | 3D analog |
|---|---|---|
| Config | `environment_config` over `(z, h)` grids | `(x, y, h)` grids, 2D defender search bounds |
| Terrain + LOS | 1D `terrain_height(z)` spline; single outward LOS sweep | 2D `terrain_height(x, y)` surface; **viewshed-style** LOS (genuinely new algorithm, not a trivial generalization of the 1D sweep) |
| Detection | radar/doppler/acoustic vs. `(z, h)` range | same formulas, Euclidean range/angles extended to `(x, y, h)` (already dimension-agnostic in form) |
| Stage cost | 4D J-map `(z, h, v, gamma)` | 6D J-map `(x, y, h, v, gamma, heading)` |
| Projection | 2D visualization-only projection | 3D visualization-only projection |
| Bellman | sweep `z` descending, `h` free | sweep `h` **ascending from goal**, `(x, y)` free |
| Attacker response | `select_authoritative_bellman_response` | same selection philosophy, 3D candidates |
| Defender search | 1D DIRECT over `z_sensor` | 2D DIRECT over `(x_sensor, y_sensor)` (`scipy.optimize.direct` already supports N-D bounds) |
| Export / Visualization | 2D `(z, h)` plots | 3D-aware: slices at fixed `h`, or `(x, y)` top-down projections |

## Preserved from `p1b_4D` (philosophy, not code)

- Universal result-envelope bundle shape (`primary_result` / `validation` / `metadata` / `status`) for every phase
- Bellman DP as the sole authoritative Attacker solver -- no NLP re-optimization
- Terrain treated as **one** combined surface, never per-obstacle special-cased
- Certified-global (DIRECT) search for Defender placement
- Phase-separated, export/import/visualize pipeline with explicit validation at every step

## What is genuinely new engineering (not a trivial dimension bump)

1. 2D terrain height function `h(x, y)` + a genuine 2D interpolant
2. A real **viewshed/raycast** LOS algorithm: for each candidate 3D point, march the line from the sensor to that point across the 2D terrain surface and check clearance -- this is the single largest new piece of work, not a generalization of the 1D running-minimum sweep built for `p1b_4D`
3. The switching point becomes a point on a 2D LOS boundary **surface**, not a 1D boundary line -- exhaustive enumeration still applies in principle, but over a much larger candidate set
4. `h`-primary (not `z`-primary) Bellman sweep direction -- a structural rewrite of the DP loop (same algorithm class, different axis)
5. `heading` as a free third action dimension -- larger action space
6. 2D Defender search space
7. 3D-aware visualization

## Scope boundary for this notebook (v1)

Single hill only. Get the full pipeline (config -> terrain/LOS -> detection -> stage cost -> Bellman -> Defender search -> export -> visualization) working and validated end-to-end on the simplest possible 3D case before considering multi-hill or more complex terrain, mirroring exactly how the 2D work in `p1b_4D` proceeded (single hill validated first, generalized to N hills only afterward).

## Attacker trajectory comparison across cost-weight scenarios

Solves the full pipeline (geometry is shared; detection/stage-cost/Bellman are rebuilt per scenario, since `w_pod`/`w_time` are baked into the CasADi `attacker_objective` graph) for three `(w_pod, w_time)` splits and plots all three optimal Attacker trajectories together on one interactive 3D figure.

In [1]:
import sys
from copy import deepcopy
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

_project_root = Path.cwd().parent if (Path.cwd() / "p1b_3DExtension").exists() is False else Path.cwd()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from p1b_3DExtension.configuration import build_configuration_bundle
from p1b_3DExtension.geometry import build_geometry_bundle
from p1b_3DExtension.detection import build_symbolic_detection_bundle
from p1b_3DExtension.stage_cost import construct_stage_cost_6d
from p1b_3DExtension.bellman import generate_bellman_candidates, select_authoritative_bellman_response


def configuration_with_weights(base_bundle, w_pod, w_time):
    """Return a ConfigurationBundle with only cost_config.attacker's weights overridden.

    Mirrors stackelberg_solver.py's _configuration_for_sensor pattern: deep-copy
    the one section being changed, leave everything else shared.
    """
    primary = base_bundle["primary_result"]
    cost_config = deepcopy(primary["cost_config"])
    cost_config["attacker"]["w_pod"] = w_pod
    cost_config["attacker"]["w_time"] = w_time
    return {**base_bundle, "primary_result": {**primary, "cost_config": cost_config}}


def solve_attacker_for_weights(base_config, w_pod, w_time):
    """Run geometry -> detection -> stage_cost -> Bellman -> response for one weight split.

    Keeps enough of the intermediate bundles (detection functions, the J6D
    glide_detection_rate component map, the coarse_step_count derived inside
    Bellman, and the winning candidate's per-step action_indices/segment_
    fractions) to let later cells reconstruct hazard/PoD-vs-time and LOS
    visibility along the path without re-solving anything.
    """
    config = configuration_with_weights(base_config, w_pod, w_time)
    geometry = build_geometry_bundle(config)
    detection = build_symbolic_detection_bundle(config, geometry)
    stage_cost = construct_stage_cost_6d(config, geometry, detection)
    bellman = generate_bellman_candidates(config, geometry, detection, stage_cost)
    response = select_authoritative_bellman_response(bellman, config)
    assert response["status"]["success"], response["status"]["message"]
    best = response["primary_result"]
    ordering = bellman["primary_result"]["cost_to_go_primary_ordering"]
    coarse_step_count = bellman["primary_result"]["bellman_diagnostics"][ordering]["coarse_step_count"]
    vehicle = config["primary_result"]["vehicle_config"]
    return {
        "w_pod": w_pod,
        "w_time": w_time,
        "trajectory": np.asarray(best["trajectory"]),
        "powered_path": np.asarray(best["powered_path"]),
        "mission_cost": best["mission_cost"],
        "mission_pod": best["mission_pod"],
        "switching_point": np.asarray(best["switching_point"]),
        "powered_time": best["powered_time"],
        "glide_time": best["glide_time"],
        "action_indices": best["metadata"]["action_indices"],
        "segment_fractions": best["metadata"]["segment_fractions"],
        "coarse_step_count": coarse_step_count,
        "geometry": geometry,
        "detection": detection,
        "grids": stage_cost["primary_result"]["grids"],
        "glide_detection_rate": stage_cost["primary_result"]["component_maps"]["glide_detection_rate"],
        "vehicle_time_step": vehicle["time_step"],
        "powered_speed": vehicle["powered_speed"],
    }


In [2]:
base_config = build_configuration_bundle()

scenarios = [
    (0.50, 0.50, "#2f6fb2"),  # blue: balanced baseline
    (0.75, 0.25, "#c0392b"),  # red: detection-averse
    (0.25, 0.75, "#2e9e5b"),  # green: time-averse
]

results = []
for w_pod, w_time, color in scenarios:
    print(f"Solving w_pod={w_pod}, w_time={w_time} ...")
    result = solve_attacker_for_weights(base_config, w_pod, w_time)
    result["color"] = color
    results.append(result)
    print(
        f"  mission_cost={result['mission_cost']:.4f}  "
        f"mission_pod={result['mission_pod']:.6f}  "
        f"switching_point={result['switching_point']}"
    )


2026-07-25 09:53:48,535 | INFO | stackelberg_3d | phase=Phase 1: Configuration status=started
2026-07-25 09:53:48,537 | WARNING | stackelberg_3d | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-25 09:53:48,537 | INFO | stackelberg_3d | phase=Phase 1: Configuration status=success elapsed_seconds=0.001908


Solving w_pod=0.5, w_time=0.5 ...
  mission_cost=0.5311  mission_pod=0.036712  switching_point=[275.   0. 175.]
Solving w_pod=0.75, w_time=0.25 ...
  mission_cost=0.2820  mission_pod=0.029697  switching_point=[137.5   0.  185. ]
Solving w_pod=0.25, w_time=0.75 ...
  mission_cost=0.7715  mission_pod=0.055676  switching_point=[343.75   0.   170.  ]


In [3]:
terrain = results[0]["geometry"]["primary_result"]["terrain_arrays"]

terrain_colorscale = [
    [0.00, "#1f4e42"], [0.25, "#3f7757"], [0.50, "#8a9a4e"],
    [0.72, "#c8a24a"], [0.88, "#e6c983"], [1.00, "#f5ecd6"],
]

surface = go.Surface(
    x=terrain["x"], y=terrain["y"], z=np.asarray(terrain["height"]).T,
    colorscale=terrain_colorscale, cmin=0.0, cmax=100.0, showscale=False,
    opacity=0.85, hoverinfo="skip", name="terrain",
    lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.9),
    lightposition=dict(x=-1500, y=-1500, z=2000),
)

traces = [surface]
for result in results:
    trajectory = result["trajectory"]
    powered_path = result["powered_path"]
    label = f"w_pod={result['w_pod']}, w_time={result['w_time']}"
    traces.append(go.Scatter3d(
        x=powered_path[:, 0], y=powered_path[:, 1], z=powered_path[:, 2],
        mode="lines", line=dict(color=result["color"], width=5, dash="dash"),
        name=f"{label} (powered)", legendgroup=label, showlegend=False,
        hovertemplate="x=%{x:.0f}<br>y=%{y:.0f}<br>h=%{z:.1f}<extra></extra>",
    ))
    traces.append(go.Scatter3d(
        x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
        mode="lines", line=dict(color=result["color"], width=6),
        name=label, legendgroup=label,
        hovertemplate="x=%{x:.0f}<br>y=%{y:.0f}<br>h=%{z:.1f}<extra></extra>",
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(
        xaxis=dict(title="x (m)", range=[0, 2750]),
        yaxis=dict(title="y (m)", range=[-1500, 750]),
        zaxis=dict(title="h (m)", range=[0, 200]),
        aspectmode="manual",
        aspectratio=dict(x=2750 / 1500, y=1500 / 1500, z=1.1),
        camera=dict(eye=dict(x=1.35, y=-1.7, z=0.85)),
    ),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.6)"),
    margin=dict(l=0, r=0, t=40, b=0),
    title="Attacker optimal trajectory across (w_pod, w_time) scenarios<br>"
          "<sub>solid = glide, dashed = powered</sub>",
    width=950, height=650,
)
fig


## Does the glide path actually leave y=0, and is it hiding behind the hill?

Check this directly from the trajectory data rather than guessing: report each
scenario's y-range along the glide path, and what fraction of glide-path nodes
are inside the sensor's `los_mask` (visible) vs. outside it (hidden by terrain).

In [4]:
def visibility_along_path(geometry_bundle, grids, path):
    """Nearest-grid-cell lookup into los_mask for each (x, y, h) path point."""
    los_mask = geometry_bundle["primary_result"]["los_masks"]["los_mask"]
    x_grid, y_grid, h_grid = grids["x"], grids["y"], grids["h"]
    x_idx = np.clip(np.rint((path[:, 0] - x_grid[0]) / (x_grid[1] - x_grid[0])).astype(int), 0, x_grid.size - 1)
    y_idx = np.clip(np.rint((path[:, 1] - y_grid[0]) / (y_grid[1] - y_grid[0])).astype(int), 0, y_grid.size - 1)
    h_idx = np.clip(np.rint((path[:, 2] - h_grid[0]) / (h_grid[1] - h_grid[0])).astype(int), 0, h_grid.size - 1)
    return los_mask[x_idx, y_idx, h_idx]


for result in results:
    trajectory = result["trajectory"]
    visible = visibility_along_path(result["geometry"], result["grids"], trajectory)
    print(
        f"w_pod={result['w_pod']}: glide y range [{trajectory[:, 1].min():.1f}, "
        f"{trajectory[:, 1].max():.1f}] m  |  "
        f"visible nodes = {int(visible.sum())}/{visible.size} "
        f"({visible.mean():.1%}) -- {'MOSTLY VISIBLE' if visible.mean() > 0.5 else 'MOSTLY HIDDEN'}"
    )


w_pod=0.5: glide y range [-800.0, 0.0] m  |  visible nodes = 5/34 (14.7%) -- MOSTLY HIDDEN
w_pod=0.75: glide y range [-850.0, 0.0] m  |  visible nodes = 5/36 (13.9%) -- MOSTLY HIDDEN
w_pod=0.25: glide y range [-750.0, 0.0] m  |  visible nodes = 5/33 (15.2%) -- MOSTLY HIDDEN


## PoD vs. time for all three scenarios

Reconstructs cumulative detection probability against mission time for each
scenario: the powered segment's acoustic hazard is trapezoidally integrated
along `powered_path` (mirrors `bellman.evaluate_powered_segment`), and the
glide segment's hazard is accumulated step by step using the winning
candidate's own recorded `action_indices`/`segment_fractions` looked up in
`glide_detection_rate` (mirrors `bellman.extract_coarse_candidate`) -- no
new computation, just replaying the same accounting the authoritative solve
already did, split out over time instead of collapsed to a final total.

In [5]:
def reconstruct_pod_vs_time(result):
    functions = result["detection"]["primary_result"]["functions"]
    sensor = result["geometry"]["primary_result"]["sensor_position"]

    # Powered segment: trapezoidal hazard integral along the sampled straight line.
    powered_path = result["powered_path"]
    n_powered = powered_path.shape[0]
    fractions = np.linspace(0.0, 1.0, n_powered)
    times_powered = fractions * result["powered_time"]
    powered_function = functions["powered_detection_components"].map(n_powered)
    outputs = powered_function(
        powered_path[:, 0].reshape(1, n_powered),
        powered_path[:, 1].reshape(1, n_powered),
        powered_path[:, 2].reshape(1, n_powered),
        np.full((1, n_powered), result["powered_speed"]),
        np.full((1, n_powered), sensor[0]),
        np.full((1, n_powered), sensor[1]),
        np.full((1, n_powered), sensor[2]),
    )
    powered_rate = np.asarray(outputs[1]).reshape(n_powered)
    cumulative_powered_hazard = np.concatenate((
        [0.0],
        np.cumsum(0.5 * (powered_rate[1:] + powered_rate[:-1]) * np.diff(times_powered)),
    ))

    # Glide segment: replay the winning candidate's own per-step actions.
    glide_rate_map = result["glide_detection_rate"]
    coarse_step_count = result["coarse_step_count"]
    time_step = result["vehicle_time_step"]
    times_glide = [result["powered_time"]]
    hazard_glide = [0.0]
    running = 0.0
    for action_key, fraction in zip(result["action_indices"], result["segment_fractions"]):
        rate = float(glide_rate_map[action_key])
        duration = coarse_step_count * time_step * fraction
        running += rate * duration
        times_glide.append(times_glide[-1] + duration)
        hazard_glide.append(running)

    times = np.concatenate((times_powered, np.asarray(times_glide[1:])))
    cumulative_hazard = np.concatenate((
        cumulative_powered_hazard,
        cumulative_powered_hazard[-1] + np.asarray(hazard_glide[1:]),
    ))
    pod = 1.0 - np.exp(-cumulative_hazard)
    return times, pod


fig2 = go.Figure()
for result in results:
    times, pod = reconstruct_pod_vs_time(result)
    label = f"w_pod={result['w_pod']}, w_time={result['w_time']}"
    fig2.add_trace(go.Scatter(
        x=times, y=pod, mode="lines", line=dict(color=result["color"], width=3), name=label,
    ))
    fig2.add_vline(x=result["powered_time"], line=dict(color=result["color"], dash="dot", width=1))
    print(f"{label}: reconstructed final PoD = {pod[-1]:.6f}  (authoritative mission_pod = {result['mission_pod']:.6f})")

fig2.update_layout(
    xaxis_title="mission time (s)",
    yaxis_title="cumulative probability of detection",
    title="PoD vs. time (dotted vlines = powered -> glide switch)",
    legend=dict(x=0.02, y=0.98),
    width=900, height=500,
)
fig2


w_pod=0.5, w_time=0.5: reconstructed final PoD = 0.036712  (authoritative mission_pod = 0.036712)
w_pod=0.75, w_time=0.25: reconstructed final PoD = 0.029697  (authoritative mission_pod = 0.029697)
w_pod=0.25, w_time=0.75: reconstructed final PoD = 0.055676  (authoritative mission_pod = 0.055676)
